In [1]:
!pip install -q torch transformers datasets accelerate

In [2]:
import torch
import json
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

In [3]:
dataset = load_dataset("tatsu-lab/alpaca", split="train[:5000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
def make_short(text):
    sentences = text.split(".")
    return sentences[0] + "." # keep only first 20 words

processed = []

for item in dataset:
    processed.append({
        "prompt": item["instruction"],
        "response": make_short(item["output"]),
        "original": item["output"]
    })

In [7]:
with open("train.json", "w") as f:
    json.dump(processed, f)

In [8]:
 model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [9]:
class MyDataset(Dataset):
    def __init__(self, file, tokenizer, max_len=64):
        with open(file) as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        text = f"Answer briefly and correctly:\nQ: {item['prompt']}\nA: {item['response']}"
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": enc["input_ids"].squeeze()
        }

In [10]:
train_dataset = MyDataset("train.json", tokenizer)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

In [11]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

for epoch in range(3):
    print(f"\nEpoch {epoch+1}")

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        print("Loss:", loss.item())


Epoch 1


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Streaming output truncated to the last 5000 lines.
Loss: 1.3227735757827759
Loss: 1.4330317974090576
Loss: 0.662845253944397
Loss: 0.6692352294921875
Loss: 1.5386180877685547
Loss: 1.7461674213409424
Loss: 0.8739811778068542
Loss: 0.8818022012710571
Loss: 0.32156112790107727
Loss: 0.8011584877967834
Loss: 0.7684139609336853
Loss: 1.345874309539795
Loss: 0.7076494693756104
Loss: 0.5338828563690186
Loss: 0.9846347570419312
Loss: 0.8471272587776184
Loss: 1.2526344060897827
Loss: 1.161080241203308
Loss: 0.44858866930007935
Loss: 0.36257466673851013
Loss: 0.7607465386390686
Loss: 1.2789578437805176
Loss: 1.1302316188812256
Loss: 0.4899950921535492
Loss: 1.0356030464172363
Loss: 0.8672866821289062
Loss: 1.3661531209945679
Loss: 1.1735299825668335
Loss: 0.8032514452934265
Loss: 0.9686084389686584
Loss: 1.391391634941101
Loss: 0.4649718105792999
Loss: 0.3667529821395874
Loss: 0.5550901889801025
Loss: 0.5377328991889954
Loss: 0.4602148234844208
Loss: 1.1991653442382812
Loss: 0.8900189995765686


In [13]:
model.eval()

prompt = "Answer briefly and correctly in one sentence:\nQ: What is gravity?\nA:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_new_tokens=15,
    do_sample=False,
    no_repeat_ngram_size=2,
    eos_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer briefly and correctly in one sentence:
Q: What is gravity?
A: Gravity is a measure of the speed of a projectile.


In [21]:
questions = [
    ("What is gravity?", "Gravity is a force that attracts objects with mass."),
    ("What is binary search?", "Binary search is an algorithm that repeatedly halves the search space."),
    ("What is AI?", "Artificial intelligence is the simulation of human intelligence by machines."),
    ("What is overfitting?", "Overfitting occurs when a model learns training data too well and fails on new data.")
]

In [22]:
def count_tokens(text):
    return len(tokenizer(text)["input_ids"])

def overlap_score(pred, ref):
    pred_words = set(pred.lower().split())
    ref_words = set(ref.lower().split())

    common = pred_words & ref_words

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_words)
    recall = len(common) / len(ref_words)

    return 2 * precision * recall / (precision + recall)

In [23]:
model.eval()

results = []

for q, ref in questions:
    prompt = f"Answer briefly and correctly in one sentence:\nQ: {q}\nA:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        no_repeat_ngram_size=2,
        eos_token_id=tokenizer.eos_token_id
    )

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    pred = pred.split("A:")[-1].strip()

    tokens = count_tokens(pred)
    score = overlap_score(pred, ref)

    print("\n----------------------")
    print("Q:", q)
    print("Model:", pred)
    print("Tokens:", tokens)
    print("Score:", score)

    results.append((tokens, score))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



----------------------
Q: What is gravity?
Model: Gravity is a measure of the speed of a projectile
Tokens: 11
Score: 0.35294117647058826

----------------------
Q: What is binary search?
Model: Binary search is a type of search algorithm used to
Tokens: 11
Score: 0.4210526315789474

----------------------
Q: What is AI?
Model: AI is a field of study that focuses on the
Tokens: 10
Score: 0.3157894736842105

----------------------
Q: What is overfitting?
Model: Overfitting is a type of software that allows users
Tokens: 10
Score: 0.16666666666666669


In [24]:
avg_tokens = sum(x[0] for x in results) / len(results)
avg_score = sum(x[1] for x in results) / len(results)

print("\n=== FINAL RESULT ===")
print("Average Tokens:", avg_tokens)
print("Average Score:", avg_score)


=== FINAL RESULT ===
Average Tokens: 10.5
Average Score: 0.3141124871001032
